# Tahap 3 — Case Retrieval

**Tujuan**: Temukan kasus lama yang paling mirip dengan query kasus baru.

Notebook ini mengimplementasikan **tiga pendekatan** retrieval dan membandingkannya:

| Pendekatan | Representasi | Model |
|---|---|---|
| A | TF-IDF | Cosine Similarity |
| B | TF-IDF | SVM (classification) |
| C | IndoBERT Embedding | Cosine Similarity |

---

## Penjelasan Singkat Tiap Metode

### TF-IDF (Term Frequency–Inverse Document Frequency)
TF-IDF adalah metode statistik untuk mengukur seberapa penting suatu kata dalam sebuah dokumen relatif terhadap seluruh koleksi dokumen.
- **TF** (Term Frequency): frekuensi kemunculan kata dalam satu dokumen.
- **IDF** (Inverse Document Frequency): memberi bobot lebih rendah pada kata yang muncul di banyak dokumen (kata umum seperti "dan", "atau").
- Setiap dokumen direpresentasikan sebagai **vektor numerik** berbobot TF-IDF.
- Kemiripan antar dokumen dihitung dengan **cosine similarity**.
- **Kelebihan**: cepat, ringan, tidak butuh GPU.
- **Kekurangan**: tidak memahami konteks/makna semantik kata.

### SVM (Support Vector Machine) di atas TF-IDF
SVM adalah model machine learning yang mencari **hyperplane** (batas keputusan) optimal untuk memisahkan kelas-kelas data dalam ruang fitur berdimensi tinggi.
- Di sini SVM digunakan untuk **klasifikasi** kategori amar putusan (misal: tolak kasasi / kabulkan kasasi).
- Input: vektor TF-IDF dari `ringkasan_fakta` atau `text_full`.
- Output: prediksi kelas → kasus dengan kelas sama dianggap "mirip".
- **Kelebihan**: efektif untuk data teks berdimensi tinggi, robust terhadap overfitting.
- **Kekurangan**: butuh label, tidak menghasilkan skor similarity langsung.

### IndoBERT (Bidirectional Encoder Representations from Transformers)
IndoBERT adalah model transformer yang di-pretrain pada korpus Bahasa Indonesia besar. Berbeda dengan TF-IDF yang berbasis statistik kata, BERT memahami **konteks semantik** — kata yang sama bisa punya makna berbeda tergantung konteks kalimatnya.
- Setiap teks diubah menjadi **dense embedding** (vektor 768 dimensi).
- Embedding diambil dari **[CLS] token** output layer terakhir.
- Kemiripan dihitung dengan **cosine similarity** antar embedding.
- **Kelebihan**: memahami makna semantik, lebih akurat untuk teks hukum kompleks.
- **Kekurangan**: butuh lebih banyak resource (RAM/GPU), lebih lambat.

## 0. Instalasi & Import

In [67]:
!pip install transformers torch scikit-learn pandas numpy


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import torch
from transformers import AutoTokenizer, AutoModel

import warnings
warnings.filterwarnings("ignore")

print("PyTorch version  :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device           :", DEVICE)

PyTorch version  : 2.9.1+cpu
CUDA available   : False
Device           : cpu


In [69]:
BASE_DIR         = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED_FOLDER = os.path.join(BASE_DIR, "data", "processed")
EVAL_FOLDER      = os.path.join(BASE_DIR, "data", "eval")
RESULTS_FOLDER   = os.path.join(BASE_DIR, "data", "results")

os.makedirs(EVAL_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

CSV_PATH     = os.path.join(PROCESSED_FOLDER, "cases_clean.csv")
QUERIES_PATH = os.path.join(EVAL_FOLDER, "queries.json")

print("CSV      :", CSV_PATH)
print("Queries  :", QUERIES_PATH)

CSV      : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\cases_clean.csv
Queries  : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\eval\queries.json


## 1. Load Data

In [70]:
df = pd.read_csv(CSV_PATH, encoding="utf-8")
print(f"Jumlah kasus : {len(df)}")
print(f"Kolom        : {list(df.columns)}")
df.head(3)

Jumlah kasus : 33
Kolom        : ['case_id', 'nomor_perkara', 'terdakwa', 'pasal_utama', 'pasal_lain', 'tanggal_putusan', 'ringkasan_fakta', 'amar_putusan', 'word_count', 'text_full']


,case_id,nomor_perkara,terdakwa,pasal_utama,pasal_lain,tanggal_putusan,ringkasan_fakta,amar_putusan,word_count,text_full
0,case_001,182,"m. umar kadavid, s.e.","372,374","197 AYAT,372 KUHP,374 KUHP,378 KUHP,488 UNDANG...",2026-03-04,terdakwa diajukan di depan persidangan pengadi...,- menolak permohonan kasasi dari pemohon kasas...,1663,putusan nomor 182 demi keadilan berdasarkan ke...
1,case_002,205,rosmawanti dewi juniarti binti (almarhum) darm...,374,"126 AYAT,244 UNDANG,253 AYAT,254 UNDANG,3 AYAT...",2026-03-04,terdakwa diajukan di depan persidangan pengadi..., mengabulkan permohonan kasasi dari pemohon k...,2246,putusan nomor 205 demi keadilan berdasarkan ke...
2,case_003,219,horas sianturi,372,"197 AYAT,253 AYAT,3 AYAT,361 HURUF,372 KITAB,3...",2026-03-05,terdakwa diajukan di depan persidangan pengadi...,− menolak permohonan kasasi dari pemohon kasas...,1991,putusan nomor 219 demi keadilan berdasarkan ke...


In [71]:
df["text_full"]       = df["text_full"].fillna("")
df["ringkasan_fakta"] = df["ringkasan_fakta"].fillna("")
df["amar_putusan"]    = df["amar_putusan"].fillna("")

TEKS_KOLOM = "ringkasan_fakta"  

texts    = df[TEKS_KOLOM].tolist()
case_ids = df["case_id"].tolist()

print(f"Kolom teks   : {TEKS_KOLOM}")
print(f"Contoh teks  : {texts[0][:200]}...")

Kolom teks   : ringkasan_fakta
Contoh teks  : terdakwa diajukan di depan persidangan pengadilan negeri kendari karena didakwa dengan dakwaan sebagai berikut: pertama : perbuatan terdakwa sebagaimana diatur dan diancam pidana dalam pasal 374 kuhp;...


## 2. Persiapan Label untuk SVM

SVM membutuhkan label kelas. Di sini kita membuat label otomatis dari `amar_putusan`:
- **`tolak`** → kasasi ditolak (paling umum)
- **`kabul`** → kasasi dikabulkan
- **`lainnya`** → putusan lain

In [72]:
def buat_label(amar: str) -> str:
    """Buat label kelas berdasarkan isi amar putusan."""
    amar = str(amar).lower()
    if "menolak" in amar or "tolak" in amar:
        return "tolak"
    elif "mengabulkan" in amar or "kabul" in amar:
        return "kabul"
    else:
        return "lainnya"

df["label"] = df["amar_putusan"].apply(buat_label)

print("Distribusi label:")
print(df["label"].value_counts())

Distribusi label:
label
tolak    29
kabul     4
Name: count, dtype: int64


In [73]:
le = LabelEncoder()
y  = le.fit_transform(df["label"])
print("Kelas  :", le.classes_)
print("Encoded:", y[:10])

Kelas  : ['kabul' 'tolak']
Encoded: [1 0 1 1 1 1 1 1 0 1]


## 3. Splitting Data (Train / Test)

Rasio **80:20** — 80% data latih, 20% data uji.
Dipakai untuk melatih SVM dan mengevaluasi semua model.

In [74]:
X_train_text, X_test_text, y_train, y_test, idx_train, idx_test = train_test_split(
    texts, y, df.index.tolist(),
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train : {len(X_train_text)} kasus")
print(f"Test  : {len(X_test_text)} kasus")

df_train = df.loc[idx_train].reset_index(drop=True)
df_test  = df.loc[idx_test].reset_index(drop=True)

Train : 26 kasus
Test  : 7 kasus


---
## Pendekatan A — TF-IDF + Cosine Similarity

### Cara kerja:
1. Semua teks kasus diubah menjadi matriks TF-IDF.
2. Teks query juga diubah ke vektor TF-IDF.
3. Cosine similarity dihitung antara vektor query dan semua vektor kasus.
4. Kasus dengan similarity tertinggi dikembalikan sebagai top-k hasil retrieval.

**Cosine Similarity** mengukur sudut antara dua vektor — nilainya 0 (tidak mirip) sampai 1 (identik).

In [75]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   
    min_df=2,             
    max_df=0.95,          
    sublinear_tf=True     
)

tfidf_vectorizer.fit(X_train_text)
tfidf_matrix_all   = tfidf_vectorizer.transform(texts)          
tfidf_matrix_train = tfidf_vectorizer.transform(X_train_text)   
tfidf_matrix_test  = tfidf_vectorizer.transform(X_test_text)    

print("Shape TF-IDF matrix (semua):", tfidf_matrix_all.shape)
print(f"Vocab size : {len(tfidf_vectorizer.vocabulary_)} token")

Shape TF-IDF matrix (semua): (33, 1484)
Vocab size : 1484 token


In [76]:
def retrieve_tfidf(query: str, k: int = 5) -> list:
    """
    Retrieve top-k kasus paling mirip dengan query menggunakan TF-IDF + Cosine Similarity.

    Parameter:
    ----------
    query : str  — teks pertanyaan / kasus baru
    k     : int  — jumlah kasus yang dikembalikan

    Return:
    -------
    List of dict berisi case_id dan similarity score
    """

    query_vec = tfidf_vectorizer.transform([query])

    sims = cosine_similarity(query_vec, tfidf_matrix_all).flatten()

    top_k_idx = sims.argsort()[::-1][:k]

    results = [
        {"case_id": case_ids[i], "similarity": round(float(sims[i]), 4)}
        for i in top_k_idx
    ]
    return results

In [77]:
contoh_query = texts[0][:300] 

hasil_tfidf = retrieve_tfidf(contoh_query, k=5)

print("Query (penggalan):", contoh_query[:150], "...")
print("\nTop-5 hasil TF-IDF:")
for r in hasil_tfidf:
    print(f"  {r['case_id']}  similarity={r['similarity']}")

Query (penggalan): terdakwa diajukan di depan persidangan pengadilan negeri kendari karena didakwa dengan dakwaan sebagai berikut: pertama : perbuatan terdakwa sebagaima ...

Top-5 hasil TF-IDF:
  case_001  similarity=0.1771
  case_027  similarity=0.1595
  case_009  similarity=0.1492
  case_025  similarity=0.1408
  case_022  similarity=0.1354


---
## Pendekatan B — TF-IDF + SVM

### Cara kerja:
1. Representasi vektor TF-IDF yang sama dipakai sebagai fitur input SVM.
2. SVM dilatih untuk mengklasifikasikan kasus ke kelas amar putusan (`tolak` / `kabul` / `lainnya`).
3. Untuk retrieval: query diprediksikan ke kelas tertentu, kemudian kasus-kasus dengan kelas sama diranking ulang menggunakan cosine similarity.

### Mengapa LinearSVC?
- `LinearSVC` menggunakan kernel linear — sangat efisien untuk data berdimensi tinggi seperti TF-IDF.
- Lebih cepat dari SVC dengan kernel RBF untuk dataset teks.
- Parameter `C` mengontrol trade-off antara margin maksimal dan kesalahan klasifikasi.

In [78]:
svm_model = LinearSVC(
    C=1.0,           
    max_iter=2000,   
    random_state=42
)
svm_model.fit(tfidf_matrix_train, y_train)

y_pred_svm = svm_model.predict(tfidf_matrix_test)

print("SVM selesai dilatih.")
print("\nClassification Report (SVM):")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

SVM selesai dilatih.

Classification Report (SVM):
              precision    recall  f1-score   support

       kabul       0.00      0.00      0.00         1
       tolak       0.86      1.00      0.92         6

    accuracy                           0.86         7
   macro avg       0.43      0.50      0.46         7
weighted avg       0.73      0.86      0.79         7



In [79]:
def retrieve_svm(query: str, k: int = 5) -> list:
    """
    Retrieve top-k kasus menggunakan SVM + TF-IDF.

    Strategi:
    1. Prediksi kelas query dengan SVM.
    2. Filter kasus yang memiliki kelas sama.
    3. Rank kasus tersebut dengan cosine similarity TF-IDF.
    4. Jika kelas kosong / hasil < k, tambahkan kasus lain berdasarkan similarity.
    """
    
    query_vec   = tfidf_vectorizer.transform([query])

    pred_class  = svm_model.predict(query_vec)[0]
    pred_label  = le.inverse_transform([pred_class])[0]

    same_class_idx = [i for i, c in enumerate(le.transform(df["label"])) if c == pred_class]

    sims = cosine_similarity(query_vec, tfidf_matrix_all).flatten()

    same_class_sims = [(i, sims[i]) for i in same_class_idx]
    same_class_sims.sort(key=lambda x: x[1], reverse=True)

    top_k_idx = [i for i, _ in same_class_sims[:k]]
    if len(top_k_idx) < k:
        remaining = sims.argsort()[::-1]
        for idx in remaining:
            if idx not in top_k_idx:
                top_k_idx.append(idx)
            if len(top_k_idx) == k:
                break

    results = [
        {"case_id": case_ids[i], "similarity": round(float(sims[i]), 4), "pred_label": pred_label}
        for i in top_k_idx
    ]
    return results

In [80]:
hasil_svm = retrieve_svm(contoh_query, k=5)

print("Top-5 hasil SVM:")
for r in hasil_svm:
    print(f"  {r['case_id']}  similarity={r['similarity']}  pred_label={r['pred_label']}")

Top-5 hasil SVM:
  case_001  similarity=0.1771  pred_label=tolak
  case_027  similarity=0.1595  pred_label=tolak
  case_025  similarity=0.1408  pred_label=tolak
  case_022  similarity=0.1354  pred_label=tolak
  case_031  similarity=0.1336  pred_label=tolak


---
## Pendekatan C — IndoBERT Embedding + Cosine Similarity

### Model: `indobenchmark/indobert-base-p1`
IndoBERT adalah model BERT yang dilatih dari awal menggunakan data Bahasa Indonesia (~4GB teks dari Wikipedia, berita, artikel). Cocok untuk tugas NLP Bahasa Indonesia termasuk dokumen hukum.

### Cara kerja:
1. Tokenisasi teks menggunakan IndoBERT tokenizer.
2. Forward pass melalui model transformer → output berupa tensor 768 dimensi per token.
3. Ambil representasi **[CLS] token** (token pertama) sebagai embedding dokumen.
   - [CLS] token di BERT secara desain mengaggregasi informasi seluruh kalimat.
4. Embedding disimpan untuk semua kasus (di-cache agar tidak perlu re-compute).
5. Saat ada query baru, hitung cosine similarity antara embedding query dengan semua embedding kasus.

### Catatan truncation:
IndoBERT memiliki batas **512 token**. Teks putusan yang panjang akan dipotong. Untuk mengatasi ini, kita pakai `ringkasan_fakta` yang lebih pendek, atau truncate otomatis di tokenizer.

In [81]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

print(f"Memuat model: {MODEL_NAME}")
print("(Proses ini membutuhkan download ~500MB pada pertama kali)")

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model     = AutoModel.from_pretrained(MODEL_NAME)
bert_model     = bert_model.to(DEVICE)
bert_model.eval()  

print(f"Model dimuat di device: {DEVICE}")

Memuat model: indobenchmark/indobert-base-p1
(Proses ini membutuhkan download ~500MB pada pertama kali)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model dimuat di device: cpu


In [82]:
def get_bert_embedding(text: str, max_length: int = 512) -> np.ndarray:
    """
    Menghasilkan sentence embedding menggunakan IndoBERT
    dengan mean pooling.
    """

    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    token_embeddings = outputs.last_hidden_state

    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_embeddings = token_embeddings * attention_mask

    sentence_embedding = (
        masked_embeddings.sum(dim=1)
        / attention_mask.sum(dim=1)
    )

    return sentence_embedding.cpu().numpy().flatten()

In [83]:
BERT_EMBED_PATH = os.path.join(PROCESSED_FOLDER, "bert_embeddings.npy")

if os.path.exists(BERT_EMBED_PATH):

    bert_embeddings = np.load(BERT_EMBED_PATH)
    print(f"Embeddings dimuat dari cache: {BERT_EMBED_PATH}")
    print(f"Shape: {bert_embeddings.shape}")
else:
    print(f"Membuat embeddings untuk {len(texts)} kasus...")
    bert_embeddings = []
    for i, text in enumerate(texts):
        emb = get_bert_embedding(text)
        bert_embeddings.append(emb)
        if (i + 1) % 5 == 0 or (i + 1) == len(texts):
            print(f"  [{i+1}/{len(texts)}] selesai")

    bert_embeddings = np.vstack(bert_embeddings)

    np.save(BERT_EMBED_PATH, bert_embeddings)
    print(f"\nEmbeddings disimpan ke: {BERT_EMBED_PATH}")
    print(f"Shape: {bert_embeddings.shape}")

Membuat embeddings untuk 33 kasus...
  [5/33] selesai
  [10/33] selesai
  [15/33] selesai
  [20/33] selesai
  [25/33] selesai
  [30/33] selesai
  [33/33] selesai

Embeddings disimpan ke: c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\bert_embeddings.npy
Shape: (33, 768)


In [84]:
def retrieve_bert(query: str, k: int = 5) -> list:
    """
    Retrieve top-k kasus paling mirip dengan query menggunakan IndoBERT + Cosine Similarity.

    Parameter:
    ----------
    query : str  — teks query / kasus baru
    k     : int  — jumlah kasus yang dikembalikan

    Return:
    -------
    List of dict berisi case_id dan similarity score
    """

    query_emb = get_bert_embedding(query).reshape(1, -1)

    sims = cosine_similarity(query_emb, bert_embeddings).flatten()

    top_k_idx = sims.argsort()[::-1][:k]

    results = [
        {"case_id": case_ids[i], "similarity": round(float(sims[i]), 4)}
        for i in top_k_idx
    ]
    return results

In [85]:
hasil_bert = retrieve_bert(contoh_query, k=5)

print("Top-5 hasil IndoBERT:")
for r in hasil_bert:
    print(f"  {r['case_id']}  similarity={r['similarity']}")

Top-5 hasil IndoBERT:
  case_003  similarity=0.7068
  case_033  similarity=0.6977
  case_029  similarity=0.6824
  case_006  similarity=0.6823
  case_031  similarity=0.6822


In [86]:
if not os.path.exists(QUERIES_PATH):
    raise FileNotFoundError(
        f"File tidak ditemukan: {QUERIES_PATH}\n"
        "Buat file queries.json terlebih dahulu di folder data/eval/\n"
        "Format: [{\"query_id\": \"q001\", \"query_text\": \"...\", \"ground_truth\": [\"case_001\"]}]"
    )

with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

print(f"Jumlah query : {len(queries)}")
for q in queries:
    print(f"  {q['query_id']} → ground_truth: {q['ground_truth']}")

Jumlah query : 8
  q001 → ground_truth: ['case_001', 'case_003', 'case_007']
  q002 → ground_truth: ['case_002', 'case_005', 'case_010']
  q003 → ground_truth: ['case_004', 'case_008', 'case_012']
  q004 → ground_truth: ['case_006', 'case_011', 'case_015']
  q005 → ground_truth: ['case_009', 'case_013', 'case_017']
  q006 → ground_truth: ['case_014', 'case_016', 'case_020']
  q007 → ground_truth: ['case_018', 'case_022', 'case_025']
  q008 → ground_truth: ['case_019', 'case_023', 'case_028']


### Jalankan Semua Query pada Semua Model

In [87]:
K = 5 

retrieval_results = [] 

for q in queries:
    qid   = q["query_id"]
    qtext = q["query_text"]
    gt    = q["ground_truth"]

    res_tfidf = retrieve_tfidf(qtext, k=K)
    res_svm   = retrieve_svm(qtext, k=K)
    res_bert  = retrieve_bert(qtext, k=K)

    retrieval_results.append({
        "query_id":        qid,
        "ground_truth":    gt,
        "tfidf_top_k":     [r["case_id"] for r in res_tfidf],
        "svm_top_k":       [r["case_id"] for r in res_svm],
        "bert_top_k":      [r["case_id"] for r in res_bert],
    })

    print(f"\n[{qid}]")
    print(f"  Ground truth : {gt}")
    print(f"  TF-IDF top-k : {[r['case_id'] for r in res_tfidf]}")
    print(f"  SVM    top-k : {[r['case_id'] for r in res_svm]}")
    print(f"  BERT   top-k : {[r['case_id'] for r in res_bert]}")

retrieval_path = os.path.join(EVAL_FOLDER, "retrieval_results.json")
with open(retrieval_path, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, ensure_ascii=False, indent=2)
print(f"\nHasil retrieval disimpan ke: {retrieval_path}")


[q001]
  Ground truth : ['case_001', 'case_003', 'case_007']
  TF-IDF top-k : ['case_024', 'case_027', 'case_025', 'case_022', 'case_014']
  SVM    top-k : ['case_024', 'case_027', 'case_025', 'case_022', 'case_014']
  BERT   top-k : ['case_031', 'case_003', 'case_006', 'case_033', 'case_001']

[q002]
  Ground truth : ['case_002', 'case_005', 'case_010']
  TF-IDF top-k : ['case_020', 'case_010', 'case_002', 'case_012', 'case_031']
  SVM    top-k : ['case_020', 'case_010', 'case_031', 'case_001', 'case_013']
  BERT   top-k : ['case_031', 'case_003', 'case_033', 'case_015', 'case_006']

[q003]
  Ground truth : ['case_004', 'case_008', 'case_012']
  TF-IDF top-k : ['case_003', 'case_028', 'case_033', 'case_015', 'case_032']
  SVM    top-k : ['case_003', 'case_028', 'case_033', 'case_015', 'case_032']
  BERT   top-k : ['case_003', 'case_033', 'case_015', 'case_029', 'case_031']

[q004]
  Ground truth : ['case_006', 'case_011', 'case_015']
  TF-IDF top-k : ['case_010', 'case_024', 'case_01